In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Paths -- edit these for your environment.
RAW_DIR  = "/PATH/TO/raw"              # raw MGB extracts (edw/, clarity/, Datamart/)
DATA_DIR = "/PATH/TO/data"             # project data root; lookup tables live here
WORK_DIR = f"{DATA_DIR}/20250604"      # identified intermediate outputs -- keep on the secure server
DEID_DIR = f"{WORK_DIR}/deidentified"  # de-identified outputs and ID / date-shift maps

In [ ]:
base_cohort = pd.read_csv(f'{WORK_DIR}/microbiology_cultures_cohort.csv')
mrn_map = pd.read_csv(f'{RAW_DIR}/edw/MRN_map/EDW_MRN_map_20240713.csv')
micro = pd.read_csv(f'{RAW_DIR}/Datamart/AllMGB_micro_20150525-20250201.csv')

In [ ]:
base_cohort = base_cohort[['anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded', 'order_time_jittered_utc']].copy()
base_cohort = base_cohort.drop_duplicates().reset_index(drop=True)
base_cohort_mrn = base_cohort.merge(mrn_map,
                                    left_on = ['anon_id'],
                                    right_on = ['PatientID'])
base_cohort_mrn

In [ ]:
## TODO:
Frequent_ABX = []

In [ ]:
micro = micro[micro['pheno_merged'] == 'Resistant'].copy()
# micro = micro[micro['drug'].isin(Frequent_ABX)]

In [ ]:
cohort_with_micro = base_cohort_mrn.merge(micro,
                                          on = ['hospital','MRN'],
                                          how = 'inner')

cohort_with_micro['resistant_time_to_cultureTime'] = (pd.to_datetime(cohort_with_micro['order_time_jittered_utc']) - 
                                                      pd.to_datetime(cohort_with_micro['coll_datetime'])).dt.days

cohort_with_micro = cohort_with_micro[cohort_with_micro['resistant_time_to_cultureTime'] > 0]

cohort_with_micro = cohort_with_micro[['anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded', 'order_time_jittered_utc',
                                       'org_clean','drug','result_datetime','resistant_time_to_cultureTime']]

cohort_with_micro = cohort_with_micro.rename(columns={'org_clean':'organism',
                                                      'drug':'antibiotic',
                                                      'result_datetime':'recorded_resistsant_time'})
cohort_with_micro

In [ ]:
microbiology_microbial_resistance_augmented = base_cohort.merge(cohort_with_micro,
                                                                on = ['anon_id','pat_enc_csn_id_coded','order_proc_id_coded','order_time_jittered_utc'],
                                                                how = 'left')
microbiology_microbial_resistance_augmented = microbiology_microbial_resistance_augmented.drop_duplicates().reset_index(drop=True)
microbiology_microbial_resistance_augmented

In [ ]:
microbiology_microbial_resistance_augmented.to_csv(f'{WORK_DIR}/microbiology_cultures_microbial_resistance_augmented.csv',
                                                   index=False)

## Deidentification

In [ ]:
# Load the saved mappings (assume you saved as CSVs before)
anon_map_df = pd.read_csv(f'{DEID_DIR}/anon_id_map.csv')
enc_map_df = pd.read_csv(f'{DEID_DIR}/enc_id_map.csv')
order_proc_map_df = pd.read_csv(f'{DEID_DIR}/order_proc_id_map.csv')

In [ ]:
# Convert to Dict
anon_id_map = dict(zip(anon_map_df['anon_id'], anon_map_df['anon_id_deid']))
enc_id_map = dict(zip(enc_map_df['pat_enc_csn_id_coded'], enc_map_df['pat_enc_csn_id_deid']))
order_proc_map = dict(zip(order_proc_map_df['order_proc_id_coded'], order_proc_map_df['order_proc_id_deid']))

# Apply the mappings
final_table_deid = microbiology_microbial_resistance_augmented.copy()
final_table_deid['anon_id'] = final_table_deid['anon_id'].map(anon_id_map)
final_table_deid['pat_enc_csn_id_coded'] = final_table_deid['pat_enc_csn_id_coded'].map(enc_id_map)
final_table_deid['order_proc_id_coded'] = final_table_deid['order_proc_id_coded'].map(order_proc_map)

final_table_deid

In [ ]:
final_table_deid.to_csv(f'{DEID_DIR}/microbiology_cultures_microbial_resistance_augmented.csv',
                        index = False)